# Task 3 (Reproduction): DDRNet-23-slim on CamVid

Reproducing **DDRNet-23-slim** on CamVid (`960x720`, 11 classes), verifying both the official checkpoint (`DDRNet23s_camvid.pth`) and an 8-epoch CPU scratch run.

In [ ]:
import os, sys, json, subprocess
from pathlib import Path

# Auto-detect Kaggle environment
IS_KAGGLE = "KAGGLE_KERNEL_RUN_TYPE" in os.environ or Path("/kaggle").exists()

if IS_KAGGLE:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "yacs", "thop"])
    repo_dest = Path("/kaggle/working/AI512_Project")
    if not repo_dest.exists():
        subprocess.run(["git", "clone", "https://github.com/Gitansh-Bansal/AI512_Project.git", str(repo_dest)])
    else:
        subprocess.run(["git", "-C", str(repo_dest), "pull"])
    os.chdir(str(repo_dest))

# Invalidate cached modules
for mod_name in list(sys.modules.keys()):
    if any(prefix in mod_name for prefix in ['train_camvid', 'test_camvid', 'datasets.camvid', 'DDRNet_23_slim']):
        del sys.modules[mod_name]

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from PIL import Image
import torch

CWD = Path(".").resolve()
REPO_ROOT = next((p for p in [CWD] + list(CWD.parents) if (p / "DDRNet").exists()), CWD)
DDRNET_DIR = REPO_ROOT / "DDRNet"
for p in [str(DDRNET_DIR), str(DDRNET_DIR / "segmentation")]:
    if p not in sys.path:
        sys.path.insert(0, p)

from DDRNet_23_slim import DualResNet, BasicBlock
from datasets.camvid import CamVid
from benchmarks import get_hardware_info, profile_parameters, profile_flops, benchmark_inference_speed
from train_camvid import resolve_camvid_root, run_training, plot_training_history
from test_camvid import run_testing

# Auto-locate CamVid dataset in /kaggle/input or local
DATA_ROOT = None
if Path("/kaggle/input").exists():
    for f in Path("/kaggle/input").rglob("0001TP_*.png"):
        for p_cand in f.parents:
            if (p_cand / "images").exists() or (p_cand / "camvid").exists() or (p_cand / "CamVid").exists():
                DATA_ROOT = p_cand
                break
        if DATA_ROOT:
            break

if DATA_ROOT is None:
    DATA_ROOT = resolve_camvid_root()

print(f"REPO_ROOT: {REPO_ROOT}")
print(f"DATA_ROOT: {DATA_ROOT}")

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {device} | PyTorch: {torch.__version__}")


## 1. Model, Complexity & Speed

In [ ]:
model = DualResNet(BasicBlock, [2, 2, 2, 2], num_classes=11, planes=32, spp_planes=128, head_planes=64, augment=False).to(device).eval()

with torch.no_grad():
    out = model(torch.randn(1, 3, 720, 960, device=device))
print("Input: (1, 3, 720, 960) -> Output:", tuple(out.shape))

param_info = profile_parameters(model)
flop_profiles = profile_flops(model, [(720, 960)], device=device)
hw_info = get_hardware_info(device)
speed = benchmark_inference_speed(model, input_resolution=(720, 960), batch_size=1, device=device, warmup_runs=5, timed_runs=15)

print(f"Params: {param_info['total_params_M']:.2f} M | Hardware: {hw_info['CPU_Processor']}")
print(f"FLOPs (960x720): {flop_profiles[0]['FLOPs_G']:.2f} G | Latency: {speed['Mean_Latency_ms']:.1f} ms | FPS: {speed['FPS']:.2f}")

## 2. Training (CamVid, 8 Epochs CPU Scratch)

In [ ]:
TRAIN_CONFIG = {
    "epochs": 10 if torch.cuda.is_available() else 4,
    "batch_size": 4 if torch.cuda.is_available() else 2,
    "optimizer": "adamw",
    "base_lr": 0.001,
    "aux_weight": 0.8,
    "crop_size": (288, 384),
    "val_freq": 1,
    "val_samples": None if torch.cuda.is_available() else 50,
    "save_dir": str(REPO_ROOT / "output" / "camvid_ddrnet_train"),
}

RETRAIN = True if (IS_KAGGLE and torch.cuda.is_available()) else False
history_path = Path(TRAIN_CONFIG["save_dir"]) / "training_history.json"
ckpt_path = Path(TRAIN_CONFIG["save_dir"]) / "best_ddrnet_camvid.pt"

if RETRAIN or not (history_path.exists() and ckpt_path.exists()):
    history = run_training(
        data_root=str(DATA_ROOT),
        train_list="list/camvid/train.lst",
        val_list="list/camvid/val.lst",
        epochs=TRAIN_CONFIG["epochs"],
        batch_size=TRAIN_CONFIG["batch_size"],
        optimizer_name=TRAIN_CONFIG["optimizer"],
        base_lr=TRAIN_CONFIG["base_lr"],
        aux_weight=TRAIN_CONFIG["aux_weight"],
        crop_h=TRAIN_CONFIG["crop_size"][0],
        crop_w=TRAIN_CONFIG["crop_size"][1],
        val_freq=TRAIN_CONFIG["val_freq"],
        val_samples=TRAIN_CONFIG["val_samples"],
        save_dir=TRAIN_CONFIG["save_dir"],
    )
else:
    history = json.loads(history_path.read_text())
    print(f"Loaded {len(history)}-epoch history from {history_path.name}")

plot_training_history(history)
plt.show()


## 3. Test Set Evaluation (Official Checkpoint vs. CPU Scratch, N=233)

In [ ]:
official_ckpt = DDRNET_DIR / "pretrained_models" / "DDRNet23s_camvid.pth"
official_out_dir = REPO_ROOT / "output" / "camvid_ddrnet_official_test"
trained_out_dir = REPO_ROOT / "output" / "camvid_ddrnet_test"

official_json = official_out_dir / "test_metrics_summary.json"
official_summary = json.loads(official_json.read_text()) if official_json.exists() else run_testing(
    data_root=DATA_ROOT, test_list="list/camvid/test.lst",
    weights_path=str(official_ckpt), output_dir=str(official_out_dir), num_samples=None, official_mode=True,
)

trained_summary = run_testing(
    data_root=DATA_ROOT, test_list="list/camvid/test.lst",
    weights_path=str(ckpt_path), output_dir=str(trained_out_dir), batch_size=2, num_samples=None, official_mode=False,
)

display(pd.DataFrame([
    {"Run": "Official DDRNet23s_camvid.pth", "mIoU (%)": official_summary["mIoU_percent"], "Dice (%)": official_summary["mean_Dice_percent"], "Sensitivity (%)": official_summary["mean_Sensitivity_percent"], "Specificity (%)": official_summary["mean_Specificity_percent"], "Pixel Acc (%)": official_summary["Pixel_Accuracy_percent"]},
    {"Run": "CPU 8-Epoch Scratch", "mIoU (%)": trained_summary["mIoU_percent"], "Dice (%)": trained_summary["mean_Dice_percent"], "Sensitivity (%)": trained_summary["mean_Sensitivity_percent"], "Specificity (%)": trained_summary["mean_Specificity_percent"], "Pixel Acc (%)": trained_summary["Pixel_Accuracy_percent"]},
]))

## 4. Per-Class Breakdown & Sample Predictions

In [ ]:
df_tr = pd.read_csv(trained_out_dir / "test_per_class_metrics.csv")
df_off = pd.read_csv(official_out_dir / "test_per_class_metrics.csv")
display(pd.DataFrame({
    "Class": df_tr["Class_Name"],
    "Pixels_GT": df_tr["Pixels_GT"],
    "CPU_8Ep_IoU (%)": df_tr["IoU (%)"].round(2),
    "Official_IoU (%)": df_off["IoU (%)"].round(2),
    "Official_Dice (%)": df_off["Dice (%)"].round(2),
}))

test_ds = CamVid(root=DATA_ROOT, list_path="list/camvid/test.lst", num_classes=11, multi_scale=False, flip=False, base_size=960, crop_size=(720, 960))
fig, axes = plt.subplots(3, 4, figsize=(18, 10))
for i, item in enumerate(test_ds.files[:3]):
    raw_img = np.array(Image.open(os.path.join(DATA_ROOT, "camvid", item["img"])).convert("RGB"))
    gt_col = np.array(Image.open(os.path.join(DATA_ROOT, "camvid", item["label"])).convert("RGB"))
    pred_cpu = np.array(Image.open(trained_out_dir / "predicted_masks" / f"{item['name']}_pred.png"))
    pred_off = np.array(Image.open(official_out_dir / "predicted_masks" / f"{item['name']}_pred.png"))
    axes[i, 0].imshow(raw_img); axes[i, 0].set_title(item["name"]); axes[i, 0].axis("off")
    axes[i, 1].imshow(gt_col); axes[i, 1].set_title("Ground Truth"); axes[i, 1].axis("off")
    axes[i, 2].imshow(pred_cpu); axes[i, 2].set_title("CPU 8-Epoch"); axes[i, 2].axis("off")
    axes[i, 3].imshow(pred_off); axes[i, 3].set_title("Official Ckpt"); axes[i, 3].axis("off")
plt.tight_layout()
plt.show()

## 5. Paper vs. Reproduction Comparison

In [ ]:
display(pd.DataFrame({
    "Metric": ["Params", "Test mIoU (N=233)", "Pixel Accuracy", "Speed"],
    "Paper (Hong et al., 2022)": ["5.70 M", "74.7% (ImageNet) / 78.6% (Cityscapes)", "?", "230 FPS (RTX 2080Ti)"],
    "Official Ckpt (CPU)": [f"{param_info['total_params_M']:.2f} M", f"{official_summary['mIoU_percent']:.2f}%", f"{official_summary['Pixel_Accuracy_percent']:.2f}%", f"{speed['FPS']:.2f} FPS (CPU)"],
    "CPU 8-Epoch Scratch": [f"{param_info['total_params_M']:.2f} M", f"{trained_summary['mIoU_percent']:.2f}%", f"{trained_summary['Pixel_Accuracy_percent']:.2f}%", f"{speed['FPS']:.2f} FPS (CPU)"],
}))